In [6]:
"""
Comparing Stata's _rmcoll vs statsmodels collinearity handling
Three test cases:
  1. Exact collinearity    (x3 = 2*x1 + x2)
  2. Near collinearity     (x3 ≈ x1 + x2 + tiny noise)
  3. Dummy variable trap   (d1 + d2 + d3 = 1 = constant)
"""

import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy.linalg import qr

np.random.seed(42)
n = 200

# ── Data ────────────────────────────────────────────────────────────────────
x1 = np.random.randn(n)
x2 = np.random.randn(n)
y  = 2*x1 - 0.5*x2 + np.random.randn(n)

# Case 1: exact collinearity
x3_exact = 2*x1 + x2

# Case 2: near collinearity (tiny noise)
x3_near  = x1 + x2 + np.random.randn(n) * 1e-10

# Case 3: dummy trap (3 dummies that sum to 1)
groups   = np.random.choice([0,1,2], size=n)
d1 = (groups == 0).astype(float)
d2 = (groups == 1).astype(float)
d3 = (groups == 2).astype(float)   # d1+d2+d3 = 1 always

data       = pd.DataFrame(np.column_stack([y, x1, x2, x3_exact, x3_near]))
data.columns = ['y', 'x1', 'x2', 'x3_exact', 'x3_near']
data.to_stata('/Users/anzony.quisperojas/Documents/GitHub/DCDH_TODO/_data/data_test.dta')


In [7]:

# ── Pivoted QR — replicating _rmcoll logic ───────────────────────────────────
def rmcoll(X, varnames, tol=1e-7):
    """
    Replicates Stata _rmcoll:
    - Pivoted QR decomposition
    - Drop column j if |R[j,j]| / |R[0,0]| < tol
    Returns: kept column indices, kept variable names, R diagonal ratios
    """
    Q, R, P = qr(X, pivoting=True)          # X[:,P] = Q @ R
    diag    = np.abs(np.diag(R))
    ratios  = diag / diag[0] if diag[0] > 0 else diag

    kept_pivoted = np.where(ratios >= tol)[0]   # positions in pivoted order
    kept_original = np.sort(P[kept_pivoted])     # back to original column order

    dropped_pivoted  = np.where(ratios < tol)[0]
    dropped_original = P[dropped_pivoted]

    print(f"  R diagonal ratios: {np.round(ratios, 2)}")
    if len(dropped_original) > 0:
        for d in dropped_original:
            print(f"  [_rmcoll] note: {varnames[d]} omitted because of collinearity")
    else:
        print("  [_rmcoll] no collinearity detected")

    return kept_original, [varnames[i] for i in kept_original]


# ── statsmodels OLS — how it handles collinearity ───────────────────────────
def sm_ols(X, y, varnames):
    """
    Fits OLS with statsmodels.
    Reports NaN coefficients (statsmodels behaviour) and effective rank.
    """
    model  = sm.OLS(y, X).fit()
    params = model.params
    print(f"  [statsmodels] effective rank: {model.df_model + 1:.0f} / {X.shape[1]}")
    for name, coef in zip(varnames, params):
        status = "NaN ← collinear" if np.isnan(coef) else f"{coef:.4f}"
        print(f"    {name:15s}  coef = {status}")
    return model



In [24]:
class _SvdWlsResult:
    """Result of SVD-based WLS that handles rank-deficient design matrices.

    Stata's `reg` uses Gauss-Jordan elimination and automatically drops collinear
    variables. With exact_match + high polynomial order, the design matrix is
    often rank-deficient in float32. This class mimics Stata's behavior by using
    SVD with a rank tolerance matching float32 precision (~1e-7).
    """

    def __init__(self, beta: np.ndarray, col_names: list, intercept_idx: int,
                 formula: str, condition_number: float = np.inf):
        self.params = pd.Series(beta, index=col_names)
        self._beta = beta
        self._col_names = col_names
        self._intercept_idx = intercept_idx
        self._formula = formula
        self.condition_number = condition_number

    def predict(self, df_new: pd.DataFrame) -> np.ndarray:
        # Build design matrix manually to handle NaN rows (patsy drops them)
        n = len(df_new)
        X_cols = []
        for col_name in self._col_names:
            if col_name == "Intercept":
                X_cols.append(np.ones(n))
            elif col_name in df_new.columns:
                X_cols.append(df_new[col_name].to_numpy(dtype=float))
            else:
                X_cols.append(np.full(n, np.nan))
        X = np.column_stack(X_cols)
        return X @ self._beta  # NaN in X → NaN in prediction


In [ ]:
def _svd_wls(formula: str, df: pd.DataFrame, weights: pd.Series,
             rcond: float = 1e-7, use_float32: bool = False,
             use_rmcoll: bool = False):
    """Weighted least squares via sweep (Gauss-Jordan) or SVD with rank tolerance.

    Matches Stata's `reg` behavior: drops collinear terms automatically.

    When use_float32=True (exact_match), uses Stata's sweep algorithm:
    1. Data is cast to float32 (matching Stata's default float storage)
    2. Cross-product X'X is computed in float64 (matching Stata's internal math)
    3. Sweep (Gauss-Jordan) drops individual variables whose pivot falls below
       a tolerance, matching Stata's automatic collinearity handling in `reg`.

    When use_rmcoll=True (only with use_float32=True), uses Stata's _rmcoll
    algorithm: reverse variable order (right-to-left) + c(epsfloat) tolerance.
    This matches Stata's collinearity detection for severely ill-conditioned
    polynomial designs.  The intercept is protected from being dropped.

    When use_float32=False, uses SVD-based lstsq (standard WLS).
    """
    import patsy
    y_dm, X_dm = patsy.dmatrices(formula, data=df, return_type="dataframe")
    y = y_dm.to_numpy(dtype=float).ravel()
    X = X_dm.to_numpy(dtype=float)
    # patsy may drop NaN rows; align weights with the surviving rows
    w = weights.reindex(X_dm.index).to_numpy(dtype=float)

    W_sqrt = np.sqrt(np.clip(w, 0, None))
    Xw = X * W_sqrt[:, None]
    yw = y * W_sqrt

    if use_float32:
        # Stata stores data in float (float32). Cast to match.
        Xw = Xw.astype(np.float32).astype(np.float64)
        yw = yw.astype(np.float32).astype(np.float64)
        if not np.all(np.isfinite(Xw)) or not np.all(np.isfinite(yw)):
            raise ValueError("Regression failed: float32 overflow in design matrix")

        n, p = Xw.shape

        col_names = list(X_dm.columns)
        icept_idx = None
        for ci, cn in enumerate(col_names):
            if cn == "Intercept":
                icept_idx = ci
                break

        non_icept = [i for i in range(p) if i != icept_idx] if icept_idx is not None else list(range(p))

        if use_rmcoll and icept_idx is not None:
            # _rmcoll mode: reverse variable order (right-to-left processing)
            # with c(epsfloat) tolerance, matching Stata's _rmcoll collinearity
            # detection.  Intercept is last and protected from being dropped.
            reorder = non_icept[::-1] + [icept_idx]
            _SWEEP_TOL = float(np.finfo(np.float32).eps)  # ~1.19e-7
            _protect_last = True
        elif icept_idx is not None and icept_idx < p - 1:
            # Standard mode: forward variable order, intercept last, tight tol
            reorder = non_icept + [icept_idx]
            _SWEEP_TOL = 1e-15
            _protect_last = False
        else:
            reorder = None
            _SWEEP_TOL = 1e-15
            _protect_last = False

        if reorder is not None:
            Xw = Xw[:, reorder]

        # Build augmented cross-product matrix [X'X, X'y; y'X, y'y]
        XtX = Xw.T @ Xw
        Xty = Xw.T @ yw
        yty = float(yw @ yw)
        A = np.zeros((p + 1, p + 1))
        A[:p, :p] = XtX
        A[:p, p] = Xty
        A[p, :p] = Xty
        A[p, p] = yty

        orig_diag = np.diag(XtX).copy()
        dropped = np.zeros(p, dtype=bool)

        for k in range(p):
            pivot = A[k, k]
            threshold = _SWEEP_TOL * abs(orig_diag[k])
            is_protected = _protect_last and (k == p - 1)
            if (abs(pivot) < threshold or abs(orig_diag[k]) == 0) and not is_protected:
                dropped[k] = True
                A[k, :] = 0.0
                A[:, k] = 0.0
                continue
            pivot_inv = 1.0 / pivot
            row_k = A[k, :].copy()
            col_k = A[:, k].copy()
            A -= np.outer(col_k, row_k) * pivot_inv
            A[k, :] = row_k * pivot_inv
            A[:, k] = -col_k * pivot_inv
            A[k, k] = -pivot_inv

        beta = A[:p, p].copy()
        beta[dropped] = 0.0

        # Reorder beta back to patsy's original column order (intercept first)
        if reorder is not None:
            inv_reorder = [0] * p
            for new_pos, old_pos in enumerate(reorder):
                inv_reorder[old_pos] = new_pos
            beta = beta[inv_reorder]

        rank = int(np.sum(~dropped))
        if rank == 0:
            raise ValueError("Regression failed: all variables dropped by sweep")
        cond = np.inf
    else:
        beta, _, rank, sv = np.linalg.lstsq(Xw, yw, rcond=rcond)
        cond = float(sv[0] / sv[-1]) if len(sv) > 0 and sv[-1] > 0 else np.inf

    return  (beta, list(X_dm.columns), 0, formula, cond)
    

In [31]:
_svd_wls(formula = "y ~ x3_exact + x2 + x1 ", df = data, weights = pd.Series(np.repeat(1, data.shape[0]) ), use_float32=True, use_rmcoll=True)

(array([-0.08968089,  0.        , -0.52018157,  1.85875232]),
 ['Intercept', 'x3_exact', 'x2', 'x1'],
 0,
 'y ~ x3_exact + x2 + x1 ',
 inf)

In [22]:
data.columns

Index(['y', 'x1', 'x2', 'x3_exact', 'x3_near'], dtype='object')

In [ ]:

def rmcoll(X, varnames, tol=1e-7):
    """
    Correct _rmcoll replica using PIVOTED QR (order-invariant).

    The pivot reorders columns by decreasing marginal norm at each step.
    The R diagonal measures each column's marginal contribution AFTER
    removing the projection onto all previously selected columns.

    A column is dropped if:
        |R[j,j]| / |R[0,0]|  <  tol  (~1e-7 in Stata)

    Because pivoting is norm-based (not position-based), the same
    variable is always dropped regardless of input column order.
    """
    Q, R, P = qr(X, pivoting=True)          # X[:, P] = Q @ R
    diag    = np.abs(np.diag(R))
    ratios  = diag / diag[0] if diag[0] > 0 else diag

    kept_idx    = sorted([P[i] for i, r in enumerate(ratios) if r >= tol])
    dropped_idx = sorted([P[i] for i, r in enumerate(ratios) if r <  tol])

    for i in dropped_idx:
        pivot_pos = list(P).index(i)
        print(f"  [_rmcoll] '{varnames[i]}' omitted — collinear"
              f"  (ratio = {ratios[pivot_pos]:.2e})")
    if not dropped_idx:
        print("  [_rmcoll] no collinearity detected")

    return kept_idx, [varnames[i] for i in kept_idx]



In [34]:
def rmcoll(X, varnames, tol=1e-7):
    Q, R, P = qr(X, pivoting=True)
    diag   = np.abs(np.diag(R))
    ratios = diag / diag[0] if diag[0] > 0 else diag

    kept_idx    = sorted([P[i] for i, r in enumerate(ratios) if r >= tol])
    dropped_idx = sorted([P[i] for i, r in enumerate(ratios) if r <  tol])

    return kept_idx, [varnames[i] for i in kept_idx]

In [ ]:
X1       = np.column_stack([x1, x2, x3_exact])
names1   = [ "x1",   "x2", "x3_exact"]
X1_const = sm.add_constant(X1)
names1_c = ["const"] + names1
X = X1_const
varnames = names1_c
tol=1e-7
Q, R, P = qr(X, pivoting=True)
diag   = np.abs(np.diag(R))
ratios = diag / diag[0] if diag[0] > 0 else diag

kept_idx    = sorted([P[i] for i, r in enumerate(ratios) if r >= tol])
dropped_idx = sorted([P[i] for i, r in enumerate(ratios) if r <  tol])


In [51]:
np.sqrt(X1_const[:, 0].sum())

np.float64(14.142135623730951)

In [52]:
P

array([3, 0, 2, 1], dtype=int32)

In [54]:

# ── Run all three cases ───────────────────────────────────────────────────────
separator = "=" * 60

# ── Case 1: Exact collinearity ───────────────────────────────────────────────
print(separator)
print("CASE 1 — Exact collinearity:  x3 = 2*x1 + x2")
print(separator)

X1       = np.column_stack([x3_exact, x1, x2])
names1   = [ 'x3_exact', "x1",   "x2"]
X1_const = sm.add_constant(X1)
names1_c = ["const"] + names1

print("\n--- Stata _rmcoll (replicated) ---")
kept_idx, kept_names = rmcoll(X1_const, names1_c)
print(f"  Kept variables: {kept_names}")

print(_svd_wls(formula = "y ~ x3_exact + x2 + x1 ", df = data, weights = pd.Series(np.repeat(1, data.shape[0]) ), use_float32=True, use_rmcoll=True))

print("\n--- statsmodels OLS ---")
sm_ols(X1_const, y, names1_c)


CASE 1 — Exact collinearity:  x3 = 2*x1 + x2

--- Stata _rmcoll (replicated) ---
  Kept variables: ['const', 'x3_exact', 'x2']
(array([-0.08968089,  0.        , -0.52018157,  1.85875232]), ['Intercept', 'x3_exact', 'x2', 'x1'], 0, 'y ~ x3_exact + x2 + x1 ', inf)

--- statsmodels OLS ---
  [statsmodels] effective rank: 3 / 4
    const            coef = -0.0897
    x3_exact         coef = 0.5329
    x1               coef = 0.7930
    x2               coef = -1.0531


In [53]:


# ── Case 2: Near collinearity ────────────────────────────────────────────────
print(f"\n{separator}")
print("CASE 2 — Near collinearity:   x3 ≈ x1 + x2 + noise(1e-10)")
print(separator)

X2       = np.column_stack([x1, x2, x3_near])
names2   = ["x1", "x2", "x3_near"]
X2_const = sm.add_constant(X2)
names2_c = ["const"] + names2

print("\n--- Stata _rmcoll (replicated, tol=1e-7) ---")
kept_idx, kept_names = rmcoll(X2_const, names2_c, tol=1e-7)
print(f"  Kept variables: {kept_names}")

print("\n--- statsmodels OLS ---")
sm_ols(X2_const, y, names2_c)



CASE 2 — Near collinearity:   x3 ≈ x1 + x2 + noise(1e-10)

--- Stata _rmcoll (replicated, tol=1e-7) ---
  Kept variables: ['const', 'x2', 'x3_near']

--- statsmodels OLS ---
  [statsmodels] effective rank: 4 / 4
    const            coef = -0.0921
    x1               coef = -1115403068.6379
    x2               coef = -1115403070.9951
    x3_near          coef = 1115403070.4874


In [ ]:


# ── Case 3: Dummy variable trap ──────────────────────────────────────────────
print(f"\n{separator}")
print("CASE 3 — Dummy trap:  d1 + d2 + d3 = 1 = constant")
print(separator)

X3       = np.column_stack([d1, d2, d3])
names3   = ["d1", "d2", "d3"]
X3_const = sm.add_constant(X3)
names3_c = ["const", "d1", "d2", "d3"]

print("\n--- Stata _rmcoll (replicated) ---")
kept_idx, kept_names = rmcoll(X3_const, names3_c)
print(f"  Kept variables: {kept_names}")

print("\n--- statsmodels OLS ---")
sm_ols(X3_const, y[:n], names3_c)


# ── Summary table ─────────────────────────────────────────────────────────────
print(f"\n{separator}")
print("SUMMARY — Stata _rmcoll vs statsmodels")
print(separator)

summary = pd.DataFrame({
    "Case": [
        "Exact collinearity",
        "Near collinearity (1e-10 noise)",
        "Dummy trap"
    ],
    "Stata _rmcoll": [
        "Drops x3_exact, warns user",
        "Drops x3_near (below tol=1e-7)",
        "Drops last dummy (d3)"
    ],
    "statsmodels": [
        "NaN coef for x3_exact",
        "May keep all (tol is adaptive)",
        "NaN coef for d3"
    ],
    "Tolerance": [
        "fixed ~1e-7",
        "fixed ~1e-7 vs adaptive n*eps*R[0,0]",
        "fixed ~1e-7"
    ]
})

print(summary.to_string(index=False))
print()